# Notebook 1 — SNOMED CT Concept Selection

Retrieves breast cancer disorder concepts (`<< 254837009`) from the Snowstorm REST API,
then computes pairwise ontological distances via the IS-A hierarchy.

**Output files**
- `data/concepts.csv` — concept_id, preferred_term, fsn
- `data/ontological_distances.csv` — n×n pairwise distance matrix

## 1a. Configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import time

SNOWSTORM_URL = "https://snowstorm.snomed.consultologist.ai"
BRANCH = "MAIN"

# Breast cancer hierarchy — 176 disorder concepts
ECL = "<< 254837009"
LIMIT = 300

## 1b. Retrieve disorder concepts

In [ ]:
def get_disorder_concepts(ecl: str, limit: int = 300) -> list[dict]:
    url = f"{SNOWSTORM_URL}/{BRANCH}/concepts"
    params = {
        "ecl": ecl,
        "semanticTag": "disorder",
        "limit": limit,
        "active": True,
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    return r.json()["items"]

concepts = get_disorder_concepts(ECL, limit=LIMIT)
print(f"Retrieved {len(concepts)} concepts")

In [ ]:
concept_ids     = [c["conceptId"] for c in concepts]
preferred_terms = [c["pt"]["term"] for c in concepts]
fsns            = [c["fsn"]["term"] for c in concepts]

df_concepts = pd.DataFrame({
    "concept_id":     concept_ids,
    "preferred_term": preferred_terms,
    "fsn":            fsns,
})

df_concepts.to_csv("data/concepts.csv", index=False)
print(df_concepts.head(10))

## 1c. Compute pairwise ontological distances

Distance via shared ancestor (LCA) approximation:
```
distance(A, B) = |ancestors(A)| + |ancestors(B)| − 2 × |ancestors(A) ∩ ancestors(B)|
```

In [ ]:
def get_ancestors(concept_id: str) -> set[str] | None:
    url = f"{SNOWSTORM_URL}/{BRANCH}/concepts/{concept_id}/ancestors"
    r = requests.get(url, params={"form": "inferred", "limit": 200})
    if r.status_code == 404:
        return None  # extension concept not present in this branch
    r.raise_for_status()
    return {c["conceptId"] for c in r.json()["items"]}

ancestor_sets = {}
skipped = []
for i, cid in enumerate(concept_ids):
    result = get_ancestors(cid)
    if result is None:
        skipped.append(cid)
    else:
        ancestor_sets[cid] = result
    if (i + 1) % 20 == 0:
        print(f"  {i + 1}/{len(concept_ids)} processed")
    time.sleep(0.05)

print(f"Done. {len(ancestor_sets)} concepts with ancestors, {len(skipped)} skipped (404).")
if skipped:
    print("Skipped IDs:", skipped)

# Filter concepts to only those with ancestor data
valid_ids = list(ancestor_sets.keys())
mask = [cid in ancestor_sets for cid in concept_ids]
concept_ids     = [cid for cid, ok in zip(concept_ids, mask) if ok]
preferred_terms = [t   for t,   ok in zip(preferred_terms, mask) if ok]
fsns            = [f   for f,   ok in zip(fsns, mask) if ok]
df_concepts     = df_concepts[mask].reset_index(drop=True)
print(f"Proceeding with {len(concept_ids)} concepts.")

In [ ]:
def ontological_distance(a_id, b_id):
    anc_a = ancestor_sets[a_id]
    anc_b = ancestor_sets[b_id]
    lca_depth = len(anc_a & anc_b)
    return len(anc_a) + len(anc_b) - 2 * lca_depth

n = len(concept_ids)
D_snomed = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        d = ontological_distance(concept_ids[i], concept_ids[j])
        D_snomed[i, j] = d
        D_snomed[j, i] = d

print(f"Distance matrix shape: {D_snomed.shape}")
print(f"Min: {D_snomed[D_snomed > 0].min():.1f}, Max: {D_snomed.max():.1f}, Mean: {D_snomed[D_snomed > 0].mean():.2f}")

In [ ]:
pd.DataFrame(D_snomed).to_csv("data/ontological_distances.csv", index=False, header=False)
print("Saved data/ontological_distances.csv")